# Jet image multi-class tagging

## 1. Preliminary steps

First we have to mount the drive to get the dataset and to download the code from the GitHub repository

In [ ]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository code
!git clone https://github.com/Fabio-Feruglio/Jet-Image-Tagging.git

# Move to the correct folder
%cd Jet-Image-Tagging/classification

We mainly used WandB to visualize and save training and validation loss and accuracy curves in real time. Some of the training loops allow also a local visualization with TensorBoard.

In [ ]:
# TensorBoard loader
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/JetTagging/checkpoints/tensorboard_logs

In [ ]:
# Install wandb
!pip install wandb

# Autenticate on WandB: insert your API Key
# You need to create a Google Colab secret named 'WANDB_API_KEY' visible to the notebook with your WandB API Key. 
# You can find your API Key in your WandB account settings.
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('WANDB_API_KEY'))

## 2. Tune and train the ResNet branch

In the following functions the --mini argument allows to select which version of the model is going to be trained: if it is set to True, then the optimized version is selected (in this case the miniResNEt, basically a ResNet with 8 layers), otherwise the full model (e.g. ResNet50) is used. Mini models have been optimized for an input image size of 128 x 128, full models use 299 x 299 following the choices made by Bassa et al. in their work.
Eventually, when changing model, it is important to modify the --save_dir and --resume_from paths, to avoid overwriting checkpoints.

First, we tune the network to find the best hyperparameters: the following cell starts an optuna study over 3 hyperparameters (learning rate, batch size and weight decay).

In [ ]:
# Tuning -- ResNet
!python /content/Jet-Image-Tagging/classification/src/tune.py \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/optuna_logs" \
    --model "resnet" \
    --mini False \
    --max_samples 20000 \
    --img_size 299 \
    --n_trials 20 \
    --tune_epochs 15 \
    --warmup_epochs 4

Then, we proceed with the main training of the network, possibly selecting the best hyperparameter combination. The patience argument implements early stopping if the validation loss has not improved after a certain number of epochs.

In [ ]:
# Training -- ResNet
!python /content/Jet-Image-Tagging/classification/src/train.py \
    --mode "resnet" \
    --mini False \
    --epochs 50 \
    --max_samples 150000 \
    --batch_size 64 \
    --img_size 299 \
    --lr 0.0001 \
    --weight_decay 0.00001 \
    --patience 5 \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/checkpoints" \
    --resume_from "/content/drive/MyDrive/JetTagging/checkpoints/resnet_latest.pth" 

Finally, we evaluate the network performance on the test set.

In [ ]:
# Evaluation -- ResNet
!python /content/Jet-Image-Tagging/classification/src/evaluation.py \
    --model "resnet" \
    --mini False \
    --model_path "/content/drive/MyDrive/JetTagging/checkpoints/resnet_best.pth" \
    --max_samples 150000 \
    --batch_size 64 \
    --img_size 299 \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/results" 

## 3. Tune and train the Inception branch

We follow the same workflow also for the inception path.

In [ ]:
# Tuning -- Inception
!python /content/Jet-Image-Tagging/classification/src/tune.py \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/optuna_logs" \
    --model "inception" \
    --mini False \
    --max_samples 100000 \
    --img_size 299 \
    --n_trials 15 \
    --tune_epochs 15 \
    --warmup_epochs 4

In [ ]:
# Training -- Inception
!python /content/Jet-Image-Tagging/classification/src/train.py \
    --mode "inception" \
    --mini False \
    --epochs 30 \
    --max_samples 150000 \
    --batch_size 32 \
    --img_size 299 \
    --lr 0.0001 \
    --weight_decay 0.0001 \
    --patience 5 \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/checkpoints" \
    --resume_from "/content/drive/MyDrive/JetTagging/checkpoints/inception_latest.pth" 

In [ ]:
# Evaluation -- Inception
!python /content/Jet-Image-Tagging/classification/src/evaluation.py \
    --model "inception" \
    --mini False \
    --model_path "/content/drive/MyDrive/JetTagging/checkpoints/inception_best.pth" \
    --max_samples 150000 \
    --batch_size 64 \
    --img_size 299 \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/results" 

## 4. Tune and train the Ensemble model

Now, we can proceed with the final training of the Ensemble model. The tuning is now performed over a broader set of parameters, which includes different learning rates for the backbone branches and the ensemble mlp head, the number of neurons in the hidden layer of the mlp head, and others.

In [ ]:
# Tuning -- Ensemble
!python /content/Jet-Image-Tagging/classification/src/tune_ensemble.py \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/optuna_logs" \
    --resnet_path "/content/drive/MyDrive/JetTagging/checkpoints/resnet_best.pth" \
    --inception_path "/content/drive/MyDrive/JetTagging/checkpoints/inception_best.pth" \
    --mini False \
    --max_samples 30000 \
    --img_size 299 \
    --n_trials 30 \
    --tune_epochs 10 \
    --warmup_epochs 4

In [ ]:
# Training launcher -- Ensemble
!python /content/Jet-Image-Tagging/classification/src/train_ensemble.py \
    --mini False \
    --resnet_weights "/content/drive/MyDrive/JetTagging/checkpoints/resnet_best.pth" \
    --inception_weights "/content/drive/MyDrive/JetTagging/checkpoints/inception_best.pth" \
    --epochs 20 \
    --warmup_epochs 3 \
    --max_samples 150000 \
    --batch_size 64 \
    --img_size 299 \
    --lr_mlp 0.0001 \
    --lr_backbone 0.000001 \
    --weight_decay 0.001 \
    --hidden_layer_size 128 \
    --dropout_mlp 0.38 \
    --patience 5 \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/checkpoints" \
    --resume_from "/content/drive/MyDrive/JetTagging/checkpoints/ensemble_latest.pth" 

In [ ]:
# Evaluation -- Ensemble
!python /content/Jet-Image-Tagging/classification/src/evaluation.py \
    --model "ensemble" \
    --mini False \
    --model_path "/content/drive/MyDrive/JetTagging/checkpoints/ensemble_best.pth" \
    --max_samples 150000 \
    --batch_size 64 \
    --hidden_layer_size 128 \
    --img_size 299 \
    --data_path "/content/drive/MyDrive/JetTagging/data/jet_images_299.h5" \
    --save_dir "/content/drive/MyDrive/JetTagging/results" 